# Data Exploration Notebook

Comprehensive exploration of the collected entity data with multilingual support, statistics, content retrieval, and analysis-ready dataframe exports.

## 1. Load and Initialize Data Sources

Import required modules and initialize database and client connections.

In [1]:
import sys
import os

# Add src to path for imports
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import logging

# Import from project
from src.data import EntityDatabase
from src.utils import setup_logging

# Setup logging
setup_logging(level=logging.INFO)
logger = logging.getLogger(__name__)

# Initialize database
db = EntityDatabase("../data/entities.db")
print("Database initialized successfully")

2025-12-27 15:50:53,698 - src.utils - INFO - Logging initialized
2025-12-27 15:50:53,699 - src.data.database - INFO - Database initialized at ../data/entities.db
Database initialized successfully


## 2. Load Sentence Transformer Model

Load the multilingual-e5-large model for generating embeddings.

In [3]:
from sentence_transformers import SentenceTransformer

# Load multilingual embeddings model
print("Loading SentenceTransformer model...")
model = SentenceTransformer('intfloat/multilingual-e5-large')
print("Model loaded successfully!")

Loading SentenceTransformer model...
2025-12-22 10:29:13,216 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: cuda:0
2025-12-22 10:29:13,217 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: intfloat/multilingual-e5-large
2025-12-22 10:29:13,217 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: intfloat/multilingual-e5-large
Model loaded successfully!
Model loaded successfully!


## 3. Explore Dataset Statistics

View overall database statistics including entity counts by type and language coverage.

In [2]:
stats = db.get_statistics()

print("\n=== DATASET STATISTICS ===\n")
print(f"Total Entities: {stats['total_entities']}")
print(f"Total Questions: {stats['total_questions']}")
print(f"Total Facts: {stats['total_facts']}")

print("\nEntities by Type:")
for entity_type, count in stats['by_type'].items():
    print(f"  {entity_type}: {count}")

print("\nEntities by Culture:")
for culture, count in stats['by_culture'].items():
    print(f"  {culture}: {count}")

print("\nEntity Labels by Language:")
for lang, count in stats['labels_by_language'].items():
    print(f"  {lang}: {count}")

print("\nWikipedia Texts by Language:")
for lang, count in stats['wikipedia_by_language'].items():
    print(f"  {lang}: {count}")


=== DATASET STATISTICS ===

Total Entities: 19828
Total Questions: 0
Total Facts: 105882

Entities by Type:
  person: 19828

Entities by Culture:
  Arabic: 7603
  East Asian: 4302
  Western: 7923

Entity Labels by Language:
  ar: 19828
  en: 19828
  fr: 19828

Wikipedia Texts by Language:
  ar: 18848
  en: 18199
  fr: 15402


## 4. Examine Entity Data Structure

Retrieve and inspect sample entities and their complete data structures.

In [3]:
# Get entities with English Wikipedia
entities_with_wiki = db.get_entities_with_wikipedia(language='en')
print(f"Entities with English Wikipedia: {len(entities_with_wiki)}")

if entities_with_wiki:
    print("\nSample entity:")
    print(entities_with_wiki[0])

# Get entities with all languages
entities_all_langs = db.get_entities_with_all_wikipedia()
print(f"\n\nEntities with all languages (en, fr, ar): {len(entities_all_langs)}")
if entities_all_langs:
    print("\nSample entity with all languages:")
    print(entities_all_langs[0])

Entities with English Wikipedia: 18199

Sample entity:
{'id': 1, 'wikidata_id': 'Q10738', 'type': 'person', 'culture': 'Western', 'country': 'United States', 'created_at': '2025-12-17 00:26:11'}


Entities with all languages (en, fr, ar): 14035

Sample entity with all languages:
{'id': 1, 'wikidata_id': 'Q10738', 'type': 'person', 'culture': 'Western', 'country': 'United States', 'created_at': '2025-12-17 00:26:11'}


## 5. Retrieve and Display Wikipedia Content

Extract Wikipedia articles for selected entities across different languages.

In [6]:
# Get first entity with English Wikipedia
if entities_with_wiki:
    entity_id = entities_with_wiki[0]['id']

    # Get all Wikipedia texts for this entity
    wikipedia_texts = db.get_wikipedia_texts(entity_id)

    print(f"Entity ID: {entity_id}")
    print(f"Available languages: {list(wikipedia_texts.keys())}\n")

    for lang, text_data in wikipedia_texts.items():
        print(f"=== {lang.upper()} ===")
        print(f"Title: {text_data['title']}")
        print(f"URL: {text_data['url']}")
        print(f"Summary (first 200 chars): {text_data['summary'][:200] if text_data['summary'] else 'N/A'}...")
        print()

Entity ID: 1
Available languages: ['ar', 'en', 'fr']

=== AR ===
Title: دوين جونسون
URL: https://ar.wikipedia.org/wiki/%D8%AF%D9%88%D9%8A%D9%86_%D8%AC%D9%88%D9%86%D8%B3%D9%88%D9%86
Summary (first 200 chars): دوين دوغلاس جونسون (بالإنجليزية: Dwayne Douglas Johnson) (من مواليد 2 مايو 1972) والمعروف أيضًا باسمه على الحلبة «ذا روك» (بالإنجليزية: The Rock) (الصخرة) وهو ممثل أمريكي كندي، ومنتج أفلام، ورجل أعمال...

=== EN ===
Title: Dwayne Johnson
URL: https://en.wikipedia.org/wiki/Dwayne_Johnson
Summary (first 200 chars): Dwayne Douglas Johnson (born May 2, 1972), also known by his ring name "the Rock", is an American actor and professional wrestler. He is signed to WWE, where he performs on a part-time basis. Widely r...

=== FR ===
Title: Dwayne Johnson
URL: https://fr.wikipedia.org/wiki/Dwayne_Johnson
Summary (first 200 chars): Dwayne Johnson ([ˈdweɪn ˈd͡ʒɑnsən]) est un acteur, producteur et catcheur (lutteur professionnel) américano-canadien, né le 2 mai 1972 à Hayward (Californie). Il 

## 6. Extract Entity Metadata and Facts

Examine entity properties, relationships, and structured information.

In [7]:
if entities_with_wiki:
    entity_id = entities_with_wiki[0]['id']

    # Get entity labels
    labels = db.get_entity_labels(entity_id)
    print("Entity Labels:")
    for lang, label in labels.items():
        print(f"  {lang}: {label}")

    # Get entity metadata
    metadata = db.get_metadata(entity_id)
    print("\nEntity Metadata:")
    for key, value in metadata.items():
        print(f"  {key}: {value}")

    # Get entity facts
    facts = db.get_entity_facts(entity_id)
    print(f"\nEntity Facts ({len(facts)} total):")
    for i, fact in enumerate(facts[:5], 1):  # Show first 5 facts
        print(f"  {i}. Property: {fact['property']}, Value: {fact['value']}")
        if fact['labels']:
            for lang, label in fact['labels'].items():
                print(f"     {lang}: {label}")

Entity Labels:
  ar: دوين جونسون
  en: Dwayne Johnson
  fr: Dwayne Johnson

Entity Metadata:
  num_wikipedia_languages: 102

Entity Facts (5 total):
  1. Property: birthDate, Value: 1972-05-02T00:00:00Z
  2. Property: birthPlace, Value: http://www.wikidata.org/entity/Q491114
     ar: هايوارد
     en: Hayward
     fr: Hayward
  3. Property: gender, Value: http://www.wikidata.org/entity/Q6581097
  4. Property: nationality, Value: http://www.wikidata.org/entity/Q30
     ar: الولايات المتحدة
     en: United States
     fr: États-Unis
  5. Property: occupation, Value: http://www.wikidata.org/entity/Q19841381
     ar: لاعب كرة قدم كندية
     en: Canadian football player
     fr: joueur de football canadien


## 7. Export Analysis-Ready DataFrames

Generate pandas DataFrames with different data structures for analysis.

In [8]:
# Export 1: Entities with metadata
df_analysis = db.export_entities_for_analysis()
print("DataFrame 1: Entities with Metadata")
print(f"Shape: {df_analysis.shape}")
print(f"Columns: {list(df_analysis.columns)}")
print(f"\nFirst few rows:\n{df_analysis.head()}")
print(f"\nData types:\n{df_analysis.dtypes}")

2025-12-22 10:32:22,913 - src.data.database - INFO - Exported 19828 entities for analysis
DataFrame 1: Entities with Metadata
Shape: (19828, 7)
Columns: ['id', 'wikidata_id', 'type', 'culture', 'country', 'created_at', 'num_wikipedia_languages']

First few rows:
   id wikidata_id    type  culture        country           created_at  \
0   1      Q10738  person  Western  United States  2025-12-17 00:26:11   
1   2      Q10731  person  Western  United States  2025-12-17 00:26:19   
2   3       Q1124  person  Western  United States  2025-12-17 00:26:23   
3   4       Q1744  person  Western  United States  2025-12-17 00:26:28   
4   5       Q4349  person  Western  United States  2025-12-17 00:26:33   

  num_wikipedia_languages  
0                     102  
1                      36  
2                     174  
3                     154  
4                      30  

Data types:
id                          int64
wikidata_id                object
type                       object
culture  

In [9]:
# Export 2: Entities with English Wikipedia text
df_en = db.export_entities_with_text(language='en')
print("\nDataFrame 2: Entities with English Wikipedia Text")
print(f"Shape: {df_en.shape}")
print(f"Columns: {list(df_en.columns)}")
print(f"\nFirst few rows:\n{df_en[['wikidata_id', 'type', 'label', 'title', 'text_length_chars']].head()}")

2025-12-22 10:32:49,006 - src.data.database - INFO - Exported 18199 entities with en Wikipedia text

DataFrame 2: Entities with English Wikipedia Text
Shape: (18199, 16)
Columns: ['id', 'wikidata_id', 'type', 'culture', 'country', 'created_at', 'label', 'title', 'summary', 'content', 'url', 'text_length_chars', 'text_length_words', 'num_sections', 'fetched_at', 'num_wikipedia_languages']

First few rows:
  wikidata_id    type           label           title  text_length_chars
0      Q10738  person  Dwayne Johnson  Dwayne Johnson              71533
1      Q10731  person      Mick Foley      Mick Foley             100860
2       Q1124  person    Bill Clinton    Bill Clinton              92876
3       Q1744  person         Madonna         Madonna              80078
4       Q4349  person   Jimmy Bennett   Jimmy Bennett               4025

DataFrame 2: Entities with English Wikipedia Text
Shape: (18199, 16)
Columns: ['id', 'wikidata_id', 'type', 'culture', 'country', 'created_at', 'label', 

In [10]:
# Export 3: Multilingual entities
df_multilang = db.export_entities_with_all_languages()
print("\nDataFrame 3: Entities with Multilingual Text")
print(f"Shape: {df_multilang.shape}")
print(f"Total columns: {len(df_multilang.columns)}")
print(f"\nLanguage-specific columns:")
lang_cols = [col for col in df_multilang.columns if any(lang in col for lang in ['en', 'fr', 'ar'])]
print(f"  {sorted(set([col.split('_')[1] for col in lang_cols if '_' in col]))}")
print(f"\nSample data (label and title columns):")
sample_cols = ['wikidata_id', 'type', 'label_en', 'title_en', 'label_fr', 'title_fr']
existing_cols = [col for col in sample_cols if col in df_multilang.columns]
print(df_multilang[existing_cols].head())

2025-12-22 10:32:57,819 - src.data.database - INFO - Exported 19828 entities with ['en', 'fr', 'ar'] text

DataFrame 3: Entities with Multilingual Text
Shape: (19828, 34)
Total columns: 34

Language-specific columns:
  ['ar', 'at', 'en', 'fr', 'length', 'sections']

Sample data (label and title columns):
  wikidata_id    type        label_en        title_en        label_fr  \
0      Q10738  person  Dwayne Johnson  Dwayne Johnson  Dwayne Johnson   
1      Q10731  person      Mick Foley      Mick Foley      Mick Foley   
2       Q1124  person    Bill Clinton    Bill Clinton    Bill Clinton   
3       Q1744  person         Madonna         Madonna         Madonna   
4       Q4349  person   Jimmy Bennett   Jimmy Bennett   Jimmy Bennett   

         title_fr  
0  Dwayne Johnson  
1      Mick Foley  
2    Bill Clinton  
3         Madonna  
4   Jimmy Bennett  

DataFrame 3: Entities with Multilingual Text
Shape: (19828, 34)
Total columns: 34

Language-specific columns:
  ['ar', 'at', 'en', 'fr